# 03. Cross-Camera Re-Identification (YoutuReID)

## 1. Overview
- **Task**: Cross-Camera Pedestrian Association & Global ID Tracking
- **Tool**: YoutuReID (OpenCV Zoo / Tencent Youtu Lab)
- **License**: Apache-2.0 (Permissive commercial license)
- **Embedding**: 768-dimensional L2-normalized feature vector

### System Design Highlights
1. **Quality Gating**: Low-resolution, blurry, or occluded person crops are discarded before identity bank insertion to avoid polluting the identity cluster.
2. **Exemplar Bank per Global ID**: Instead of a single averaged vector (which loses multi-view diversity), each Global ID retains an appearance bank (up to 30 exemplars/centroids).
3. **Camera Reachability Graph**: Searches are pruned based on physical building topology (e.g. Corridor 1 connects only to Lobby and Stairwell A), avoiding impossible transitions.

In [ ]:
# Import dependencies
import os
import numpy as np
import cv2
import onnxruntime as ort
import matplotlib.pyplot as plt

## 2. YoutuReID Feature Embedder (768-d)

In [ ]:
class YoutuReIDEmbedder:
    """Extracts 768-d L2-normalized identity embeddings."""

    INPUT_SIZE = (128, 256) # (W, H)
    _MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    _STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

    def __init__(self, model_path: str = None):
        self.session = None
        if model_path and os.path.exists(model_path):
            providers = ['MIGraphXExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(model_path, providers=providers)
            self.input_name = self.session.get_inputs()[0].name
        else:
            print('YoutuReID model not found. Running in simulation mode.')

    def embed(self, bgr_crop: np.ndarray) -> np.ndarray:
        if self.session is None or bgr_crop.size == 0:
            vec = np.random.randn(768).astype(np.float32)
            return vec / (np.linalg.norm(vec) + 1e-6)

        rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, self.INPUT_SIZE, interpolation=cv2.INTER_LINEAR)
        blob = (resized.astype(np.float32) / 255.0).transpose(2, 0, 1)
        blob = (blob - self._MEAN) / self._STD
        blob = np.expand_dims(blob, axis=0)
        
        feat = self.session.run(None, {self.input_name: blob})[0][0]
        return feat / (np.linalg.norm(feat) + 1e-6)

## 3. Cross-Camera Identity Manager & Spatial Topology

In [ ]:
class CrossCameraReIDManager:
    """Assigns Global IDs across multiple camera viewpoints."""

    def __init__(self, match_thresh: float = 0.67, bank_size: int = 15):
        self.match_thresh = match_thresh
        self.bank_size = bank_size
        self.global_banks = {}   # global_id -> list of 768-d unit vectors
        self.last_seen = {}      # global_id -> {'camera': str, 'timestamp': float}
        self.next_gid = 1001
        
        # Spatial graph defining allowed camera transitions
        self.camera_graph = {
            'cam_lobby': {'cam_corridor_1', 'cam_outdoor_gate'},
            'cam_corridor_1': {'cam_lobby', 'cam_canteen'},
            'cam_canteen': {'cam_corridor_1'},
            'cam_outdoor_gate': {'cam_lobby'}
        }

    def is_reachable(self, prev_cam: str, curr_cam: str) -> bool:
        if prev_cam == curr_cam:
            return True
        return curr_cam in self.camera_graph.get(prev_cam, set())

    def match_or_create(self, camera: str, crop_embedding: np.ndarray, timestamp: float) -> int:
        best_score = -1.0
        best_gid = None

        for gid, bank in self.global_banks.items():
            prev_cam = self.last_seen[gid]['camera']
            # Constrain by topology graph
            if not self.is_reachable(prev_cam, camera):
                continue

            # Calculate maximum cosine similarity against all bank exemplars
            bank_matrix = np.array(bank) # Shape: [N, 768]
            similarities = bank_matrix @ crop_embedding
            max_sim = float(np.max(similarities))

            if max_sim > best_score:
                best_score = max_sim
                best_gid = gid

        # Check if match meets confidence threshold
        if best_gid is not None and best_score >= self.match_thresh:
            # Update existing identity bank
            if len(self.global_banks[best_gid]) < self.bank_size:
                self.global_banks[best_gid].append(crop_embedding)
            self.last_seen[best_gid] = {'camera': camera, 'timestamp': timestamp}
            return best_gid, best_score, False
        else:
            # Spawn new Global ID
            new_gid = self.next_gid
            self.next_gid += 1
            self.global_banks[new_gid] = [crop_embedding]
            self.last_seen[new_gid] = {'camera': camera, 'timestamp': timestamp}
            return new_gid, 1.0, True

## 4. Cross-Camera Tracking Demonstration

In [ ]:
reid_mgr = CrossCameraReIDManager(match_thresh=0.70)

# Step 1: Person A enters 'cam_lobby'
person_a_base_feat = np.random.randn(768).astype(np.float32)
person_a_base_feat /= np.linalg.norm(person_a_base_feat)

gid_1, score_1, is_new_1 = reid_mgr.match_or_create('cam_lobby', person_a_base_feat, timestamp=100.0)
print(f"Event 1 (Lobby): Person detected -> Assigned Global ID: G{gid_1} (New: {is_new_1})")

# Step 2: Person A walks into 'cam_corridor_1' (slight appearance noise due to lighting)
person_a_corridor_feat = person_a_base_feat + np.random.normal(0, 0.08, size=768).astype(np.float32)
person_a_corridor_feat /= np.linalg.norm(person_a_corridor_feat)

gid_2, score_2, is_new_2 = reid_mgr.match_or_create('cam_corridor_1', person_a_corridor_feat, timestamp=145.0)
print(f"Event 2 (Corridor 1): Person detected -> Matched Global ID: G{gid_2} (Similarity: {score_2:.3f}, New: {is_new_2})")

# Step 3: Completely different Person B enters 'cam_lobby'
person_b_feat = np.random.randn(768).astype(np.float32)
person_b_feat /= np.linalg.norm(person_b_feat)

gid_3, score_3, is_new_3 = reid_mgr.match_or_create('cam_lobby', person_b_feat, timestamp=150.0)
print(f"Event 3 (Lobby): Person detected -> Assigned Global ID: G{gid_3} (New: {is_new_3})")